# 🔎 Week 4 hands-on: Retrieval evaluation

## Learning Objectives
In this notebook, you will learn:
1. **The corpus and the labeled queries** - load 7 help-centre documents, 26 section chunks and 50 graded queries
2. **Five retrieval metrics** - compute Precision@k, Recall@k, Hit Rate@k, MRR and nDCG, and check them by hand
3. **A retrieval benchmark** - compare bm25, concept-vector and hybrid retrievers at k = 3 and k = 5, by slice
4. **Filters and chunk size** - measure filter violations and what chunking costs in context size
5. **Breaking the metrics** - fix an exact-term miss and an nDCG above one

## Where this fits
This is the runnable companion to `week-04-retrieval-evaluation-handson.md`, using the same code as `lab.py` (a test keeps the two identical). Read `week-04-retrieval-evaluation-theory.md` first. Weeks 1 to 3 evaluated decisions and replies. This week evaluates the search that feeds the answer, and Week 5 evaluates the answer itself.

## Prerequisites
- `week-04-retrieval-evaluation-theory.md`, and Weeks 1 to 3
- The course environment from `new_course/setup.md`, selected as this notebook's kernel
- No API keys for the offline parts. The optional live-embeddings cell in Part 6 needs `openai` and `OPENAI_API_KEY`, and costs money

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, imports the shared corpus and retrievers, and names the query set and the hand rankings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Paths, the corpus and the retrievers
# ============================================================================
import dataclasses
import json
import math
import sys
from collections import defaultdict
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.retrieval import Chunk, Retriever, chunk_documents, current_public, load_documents  # noqa: E402

QUERIES = COURSE / "datasets" / "retrieval" / "policy-queries-v1.jsonl"
HAND_RANKINGS = COURSE / "weeks" / "week-04-retrieval-evaluation" / "dataset" / "hand_rankings.jsonl"
METRICS = ("precision", "recall", "hit", "mrr", "ndcg")

print(f"✅ Environment ready: {QUERIES.relative_to(COURSE)}")

---
## 📄 Part 1: The Corpus and the Labeled Queries

The corpus has 7 documents. One is the superseded 2025-06 refund policy and one is an internal playbook: both are **hard negatives** that must never reach a customer answer. Each query labels policy *sections* with graded relevance (3 answers, 2 needed context, 1 related), so the labels work for any chunking.

In [ ]:
# ============================================================================
# CORPUS AND QUERIES: Load and inspect
# ============================================================================
def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


docs = load_documents()
chunks = chunk_documents(docs)
queries = read_jsonl(QUERIES)
print(f"📄 {len(docs)} documents, {len(chunks)} section chunks, {len(queries)} queries")
print(f"⚠️ Excluded by the filter: {[c.chunk_id for c in chunks if not current_public(c)]}")
print(f"📋 Example query: {queries[45]['query']}  labels {queries[45]['relevance']}")

---
## 🧮 Part 2: Five Retrieval Metrics

Each metric answers a different question. Precision is noise, recall is coverage, hit rate is "anything at all", MRR is "the first good passage near the top", and nDCG is graded order. nDCG credits each labeled section once.

> **Key Insight**: Ranking H2 scores MRR 1.0 while the passage that answers the query sits at rank 4.

In [ ]:
# ============================================================================
# METRICS: The five functions and the per-query scorer
# ============================================================================
def chunk_grade(chunk: Chunk, labels: dict[str, int]) -> int:
    return max((labels.get(s, 0) for s in chunk.section_ids), default=0)


def precision_at_k(ranked: list[Chunk], labels: dict[str, int], k: int) -> float:
    """Share of the top k chunks that are relevant at all (grade > 0)."""
    return sum(chunk_grade(c, labels) > 0 for c in ranked[:k]) / k


def recall_at_k(ranked: list[Chunk], labels: dict[str, int], k: int) -> float:
    """Share of the labeled sections that some top-k chunk covers."""
    covered = {s for c in ranked[:k] for s in c.section_ids if s in labels}
    return len(covered) / len(labels)


def hit_rate_at_k(ranked: list[Chunk], labels: dict[str, int], k: int) -> float:
    """1.0 if any top-k chunk is relevant, else 0.0. Averaged over queries it is the hit rate."""
    return 1.0 if any(chunk_grade(c, labels) > 0 for c in ranked[:k]) else 0.0


def reciprocal_rank_at_k(ranked: list[Chunk], labels: dict[str, int], k: int) -> float:
    """1 / rank of the first relevant chunk in the top k, or 0. Averaged over queries it is MRR."""
    for rank, c in enumerate(ranked[:k], start=1):
        if chunk_grade(c, labels) > 0:
            return 1 / rank
    return 0.0


def ndcg_at_k(ranked: list[Chunk], labels: dict[str, int], k: int) -> float:
    """Discounted gain of the ranking divided by the gain of the best possible ranking.

    Each labeled section earns its gain once. Without that rule, two window chunks cut
    from the same section would both be credited and nDCG could exceed 1.
    """
    credited: set[str] = set()
    dcg = 0.0
    for rank, c in enumerate(ranked[:k], start=1):
        new = [s for s in c.section_ids if s in labels and s not in credited]
        credited.update(new)
        grade = max((labels[s] for s in new), default=0)
        dcg += (2 ** grade - 1) / math.log2(rank + 1)
    ideal = sorted(labels.values(), reverse=True)[:k]
    idcg = sum((2 ** g - 1) / math.log2(rank + 1) for rank, g in enumerate(ideal, start=1))
    return dcg / idcg if idcg else 0.0


def score_query(ranked: list[Chunk], labels: dict[str, int], k: int) -> dict[str, float]:
    return {"precision": precision_at_k(ranked, labels, k), "recall": recall_at_k(ranked, labels, k),
            "hit": hit_rate_at_k(ranked, labels, k), "mrr": reciprocal_rank_at_k(ranked, labels, k),
            "ndcg": ndcg_at_k(ranked, labels, k),
            "violations": sum(not current_public(c) for c in ranked[:k]),
            "words": sum(len(c.text.split()) for c in ranked[:k])}


print(f"✅ Defined {len(METRICS)} metrics: {METRICS}")

Work out the three hand rankings on paper first (the theory file's section 6), then run the next cell to check them.

In [ ]:
# ============================================================================
# HAND CHECK: Score the three fixed rankings
# ============================================================================
def print_hand() -> None:
    """Score the fixed rankings in dataset/hand_rankings.jsonl. Check them by hand first."""
    by_id = {c.chunk_id: c for c in chunk_documents(load_documents())}
    for row in read_jsonl(HAND_RANKINGS):
        ranked = [by_id[cid] for cid in row["ranking"]]
        print(f"{row['name']}: labels {row['relevance']}")
        for k in (3, 5):
            scores = score_query(ranked, row["relevance"], k)
            print(f"  k={k}  " + "  ".join(f"{m}={scores[m]:.3f}" for m in METRICS))


print_hand()

---
## 🔍 Part 3: A Retrieval Benchmark

The benchmark retrieves for all 50 queries and averages each metric. **BM25** matches words, the **concept vector** matches a hand-built list of concepts (a stand-in for embeddings, tuned on these queries, so it flatters itself), and **hybrid** fuses both with reciprocal rank fusion.

In [ ]:
# ============================================================================
# BENCHMARK: Run, average and group by slice
# ============================================================================
def run_benchmark(kind: str, k: int, filtered: bool = True, chunking: str = "section",
                  queries: list[dict] | None = None) -> list[dict]:
    """Retrieve for every query and score it. One row per query."""
    queries = queries or read_jsonl(QUERIES)
    retriever = Retriever(chunk_documents(load_documents(), chunking), kind)
    rows = []
    for q in queries:
        ranked = retriever.search(q["query"], k, filtered=filtered)
        rows.append({"query_id": q["query_id"], "slice": q["slice"], **score_query(ranked, q["relevance"], k)})
    return rows


def mean(rows: list[dict], metric: str) -> float:
    return sum(r[metric] for r in rows) / len(rows)


def by_slice(rows: list[dict]) -> dict[str, list[dict]]:
    groups: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        groups[r["slice"]].append(r)
    return dict(groups)


def print_metric_row(label: str, rows: list[dict], extra: str = "") -> None:
    cells = "  ".join(f"{mean(rows, m):>6.3f}" for m in METRICS)
    print(f"{label:<30} {cells}  {sum(r['violations'] for r in rows):>5}  {mean(rows, 'words'):>6.0f}{extra}")


def print_header(first: str) -> None:
    print(f"{first:<30} {'P@k':>6}  {'R@k':>6}  {'Hit':>6}  {'MRR':>6}  {'nDCG':>6}  {'viol':>5}  {'words':>6}")


def print_slices(kind: str, k: int) -> None:
    rows = run_benchmark(kind, k)
    print(f"{kind} @ {k} by slice (filtered, section chunks):")
    print_header("slice (queries)")
    for name, group in sorted(by_slice(rows).items(), key=lambda kv: mean(kv[1], "recall")):
        print_metric_row(f"{name} ({len(group)})", group)
    print_metric_row(f"overall ({len(rows)})", rows)


for kind in ("bm25", "concept"):
    print_slices(kind, 5)
    print()

BM25 loses half the paraphrase sections but ranks exact terms best (MRR 0.875 against 0.698). The averages hide both facts.

---
## ⚖️ Part 4: Filters and Chunk Size

The comparison runs every retriever at k = 3 and k = 5, then hybrid with the filter off and on, then three chunking strategies. The `viol` column counts superseded or internal chunks in the top k. The `words` column is the context size the answer writer would receive.

In [ ]:
# ============================================================================
# COMPARE: Retrievers, the metadata filter and chunking
# ============================================================================
def print_compare() -> None:
    print("Retrievers, filtered to current public policy, section chunks:")
    print_header("retriever @ k")
    for kind in ("bm25", "concept", "hybrid"):
        for k in (3, 5):
            print_metric_row(f"{kind} @ {k}", run_benchmark(kind, k))

    print("\nMetadata filter, hybrid @ 5:")
    print_header("filter")
    for filtered in (False, True):
        print_metric_row("on" if filtered else "off", run_benchmark("hybrid", 5, filtered=filtered))

    print("\nChunking, hybrid @ 5, filtered:")
    print_header("chunking")
    for chunking in ("section", "window", "document"):
        print_metric_row(chunking, run_benchmark("hybrid", 5, chunking=chunking))


print_compare()

In [ ]:
# ============================================================================
# ONE QUERY: The 60-day trap with the filter off
# ============================================================================
def print_query(query_id: str, kind: str, k: int, filtered: bool) -> None:
    q = next((q for q in read_jsonl(QUERIES) if q["query_id"] == query_id), None)
    if q is None:
        sys.exit(f"unknown query ID: {query_id}")
    retriever = Retriever(chunk_documents(load_documents()), kind)
    ranked = retriever.search(q["query"], k, filtered=filtered)
    print(f"{query_id} ({q['slice']}): {q['query']}")
    print(f"labels: {q['relevance']}")
    for rank, c in enumerate(ranked, start=1):
        flag = "" if current_public(c) else f"  <- {c.status}/{c.audience}"
        print(f"  {rank}. {c.chunk_id:<38} grade {chunk_grade(c, q['relevance'])}{flag}")
    print("  " + "  ".join(f"{m}={v:.3f}" for m, v in score_query(ranked, q["relevance"], k).items()))


print_query("q-035", "hybrid", 5, filtered=False)

The right passage is first, so the relevance metrics look perfect, and two excluded chunks still made the top 5. Count filter violations as their own hard rule.

---
## 🐛 Part 5: Breaking the Metrics

### 5.1 🐛 Drill 1: the exact rule number finds nothing
BM25 cannot match "R6" because the chunk text never contains it. The fix puts each section label into the text the retriever scores.

In [ ]:
# ============================================================================
# DRILL 1: Add section labels to the chunk text
# ============================================================================
q005 = next(q for q in queries if q["query_id"] == "q-005")
before = Retriever(chunks, "bm25").search(q005["query"], 3)
labeled = [dataclasses.replace(c, text=f"{c.chunk_id.split('#')[1]}. {c.text}") for c in chunks]
after = Retriever(labeled, "bm25").search(q005["query"], 3)
print(f"❌ Before: {[c.chunk_id for c in before]}")
print(f"✅ After:  {[c.chunk_id for c in after]}")

### 5.2 🐛 Drill 2: nDCG above one
This version gives every chunk its own grade. With window chunks, several windows from one section each earn its gain.

In [ ]:
# ============================================================================
# DRILL 2: Double credit, then the once-per-section rule
# ============================================================================
def ndcg_double_credit(ranked, labels, k):
    dcg = sum((2 ** chunk_grade(c, labels) - 1) / math.log2(rank + 1) for rank, c in enumerate(ranked[:k], start=1))
    ideal = sorted(labels.values(), reverse=True)[:k]
    idcg = sum((2 ** g - 1) / math.log2(rank + 1) for rank, g in enumerate(ideal, start=1))
    return dcg / idcg if idcg else 0.0


window = Retriever(chunk_documents(docs, "window"), "hybrid")
broken = [ndcg_double_credit(window.search(q["query"], 5), q["relevance"], 5) for q in queries]
fixed = [ndcg_at_k(window.search(q["query"], 5), q["relevance"], 5) for q in queries]
print(f"❌ Double credit: mean nDCG {sum(broken) / len(broken):.3f}, max {max(broken):.3f}")
print(f"✅ Once per section: mean nDCG {sum(fixed) / len(fixed):.3f}, max {max(fixed):.3f}")

---
## 💳 Part 6: Optional Live Embeddings (paid)

This cell is tagged `skip-execution` because it calls a paid API: two embeddings calls, a few thousand tokens. To run it by hand, install `requirements-integrations.txt`, set `OPENAI_API_KEY`, change `RUN_LIVE` to `True` and run the cell. It has been checked against `openai` 2.54.0 but not run against a model yet.

In [ ]:
# ============================================================================
# LIVE EMBEDDINGS: A real dense retriever versus hybrid (costs money)
# ============================================================================
def dense_live(model: str = "text-embedding-3-small", k: int = 5) -> None:
    """Compare a real embedding retriever with BM25 hybrid. Costs money: needs `openai` and OPENAI_API_KEY.

    One embeddings call for the 26 chunks and one for the 50 queries.
    """
    import os
    if not os.environ.get("OPENAI_API_KEY"):
        sys.exit("OPENAI_API_KEY is not set. dense-live is optional; every other command runs offline.")
    from openai import OpenAI  # requirements-integrations.txt

    client = OpenAI()
    chunks = [c for c in chunk_documents(load_documents()) if current_public(c)]
    queries = read_jsonl(QUERIES)

    def embed(texts: list[str]) -> list[list[float]]:
        return [d.embedding for d in client.embeddings.create(model=model, input=texts).data]

    def cos(a: list[float], b: list[float]) -> float:
        return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

    chunk_vecs, query_vecs = embed([c.text for c in chunks]), embed([q["query"] for q in queries])
    bm25 = Retriever(chunks, "bm25")
    dense_rows, hybrid_rows = [], []
    for q, qv in zip(queries, query_vecs):
        dense_order = sorted(range(len(chunks)), key=lambda i: -cos(qv, chunk_vecs[i]))
        fused = [0.0] * len(chunks)
        for order in (dense_order, bm25.ranked(q["query"])):
            for rank, i in enumerate(order, start=1):
                fused[i] += 1 / (60 + rank)
        hybrid_order = sorted(range(len(chunks)), key=lambda i: (-fused[i], i))
        for rows, order in ((dense_rows, dense_order), (hybrid_rows, hybrid_order)):
            rows.append({"query_id": q["query_id"], "slice": q["slice"],
                         **score_query([chunks[i] for i in order], q["relevance"], k)})
    print(f"Live embeddings ({model}), filtered, k={k}:")
    print_header("retriever")
    print_metric_row("dense", dense_rows)
    print_metric_row("dense + bm25 hybrid", hybrid_rows)


RUN_LIVE = False  # set to True to spend money on two embeddings calls
if RUN_LIVE:
    dense_live()
else:
    print("⚠️ Live embeddings skipped. Set RUN_LIVE = True to run them.")

---
## 📝 Summary

In this notebook, we learned:

### 1. The corpus and the labeled queries
- Labels name graded policy sections, so one label set serves every chunking strategy.

### 2. Five retrieval metrics
- Each metric has a blind spot. MRR stops at the first hit, and precision is capped by the number of relevant sections.

### 3. A retrieval benchmark
- No retriever wins every slice. BM25 ranks exact terms best, the concept vector wins paraphrases, and hybrid is the most even.

### 4. Filters and chunk size
- The filter removes 57 violations. Whole-document chunks reach recall 1.0 by sending 3.7 times the words.

### 5. Breaking the metrics
- Keep identifiers in the scored text, and credit each relevant section once.

### Next Steps
- **`week-05-answer-quality-and-evidence-handson.ipynb`**: grades the answers written from these chunks for faithfulness, correctness, citations and abstention.